# TWAS, Colocalization & ColocBoost

This notebook demonstrates four downstream analyses that integrate molecular QTLs with GWAS, using the toy `protocol_example` dataset. Each method is a self-contained part with its own input files and output files.

## Pipeline position & prerequisites

This is the final, **advanced-analysis** notebook in the `protocol_example` toy bundle. It consumes artifacts produced by earlier notebooks, so run them first in this order:

1. `data_preprocessing/2_genotype_preprocessing` → produces `output/plink/` (QC'd genotypes).
2. `data_preprocessing/1_phenotype_preprocessing` → produces `output/phenotype/` (molecular phenotypes by chrom).
3. `data_preprocessing/4_covariates_preprocessing` → produces `output/covariate/` (covariates + PCs).
4. `qtl_association_finemapping/1_xqtl_association` → produces `output/phenotype/phenotype_by_chrom_for_cis/` region lists.
5. `qtl_association_finemapping/2_finemapping` → produces `output/finemapping/susie_twas/twas_weights/` (the precomputed TWAS prediction weights Part 1 needs).

Within this notebook the parts also chain to each other: earlier parts write to `output/twas/`, `output/susie_coloc/`, and `output/colocboost/`, which later parts read back.

**Pre-staged external inputs.** A few reference inputs are not produced by any notebook in this bundle, so they are provided ready-made under `input/`: `input/twas/` (TWAS meta/LD-block tables), `input/LD_sketch/` (RSS LD sketch reference), `input/susie_enloc_data/` (enloc priors & GWAS blocks), and `input/colocboost/` (ColocBoost GWAS meta). The specific input files each part needs are listed at the start of that part.


## Overview

1. **TWAS / MR** — transcriptome-wide association and Mendelian randomization.
2. **xQTL–GWAS enrichment** — global enrichment parameters used as colocalization priors.
3. **Pairwise colocalization (SuSiE-coloc)** — gene-by-gene xQTL–GWAS colocalization.
4. **ColocBoost** — multi-trait colocalization (xQTL-only and joint xQTL–GWAS modes).

The input files needed for each part are listed at the start of that part.

## 1. TWAS

### 1.1 Transcriptome-Wide Association Study (TWAS)

**TWAS** is a method used to identify genes associated with complex traits by leveraging predicted gene expression levels. It combines precomputed molecular phenotype prediction weights with GWAS summary statistics to perform gene-trait association tests. 


#### 1.2 Input File Formats

- GWAS Metadata (`--gwas_meta_data`)    
A TSV file specifying study ID, chromosome, path to summary statistics, and a column mapping file.  
- LD Reference Metadata (`--ld_meta_data`)    
A TSV file with chromosome, start and end positions, and paths to LD matrix files.  
- xQTL Weight Metadata (`--xqtl_meta_data`)    
Metadata file containing gene region coordinates and paths to weight files stored in RDS format.  
- Genomic Region Definitions (`--regions`)    
A BED file defining analysis regions.  
- xQTL Type Table (`--xqtl_type_table`)    
A mapping file defining the molecular phenotype types and their associated biological context.


#### 1.3 Key Parameters

- `--rsq_cutoff 0.01`: Cross-validation R² threshold used to determine whether a gene is predictable  
- `--rsq_pval_cutoff 0.05`: Cross-validation p-value threshold  


#### 1.4 Analysis Workflow  

1). **Weight Loading**: Load TWAS prediction weights from RDS files.     
2). **Predictable Gene Identification**: Determine which genes are reliably predictable based on cross-validation performance.    
3). **TWAS Testing**: Compute TWAS Z-scores by combining predicted expression with GWAS statistics.    
4). **Mendelian Randomization (MR)**: Perform MR analysis on candidate genes.

### Input Files

| File | Produced by | Used as |
|------|-------------|---------|
| `input/twas/protocol_example.twas.gwas_meta.tsv` | toy data prep | GWAS summary-statistics metadata |
| `input/twas/protocol_example.twas.xqtl_meta.tsv` | toy data prep | xQTL weight metadata (gene region + RDS weights) |
| `input/twas/protocol_example.twas.LD_blocks.chr22.bed` | toy data prep | analysis region definitions |
| `input/twas/protocol_example.twas.data_type_table.txt` | toy data prep | xQTL molecular phenotype type table |
| `input/LD_sketch/meta.tsv` | RSS LD sketch step | LD reference metadata |
| `output/finemapping/susie_twas/twas_weights/*.univariate_twas_weights.rds` | fine-mapping (2_finemapping) | TWAS prediction weights |

Key parameters: `--rsq_cutoff 0.01` (cross-validation R² threshold for predictable genes) and `--rsq_pval_cutoff` (cross-validation p-value threshold).

The TWAS step proceeds in four stages. First it **loads** the precomputed prediction weights from the RDS files. It then identifies **predictable genes** — those whose expression is reliably imputable based on cross-validation performance (controlled by `--rsq_cutoff` and `--rsq_pval_cutoff`). For those genes it computes **TWAS Z-scores** by combining the predicted expression with the GWAS summary statistics, and finally performs **Mendelian Randomization (MR)** on the candidate genes to assess potential causal effects of expression on the trait.


To perform TWAS, you need to generate TWAS weight first.   
see: 
- exercise notebook: `data/exercise_notebook/qtl_association_finemapping/2_finemapping.ipynb`
- pipeline: `pipeline/mnm_regression.ipynb susie_twas`

### **Step 1.** Load the TWAS prediction weights


In [16]:
# Load the TWAS prediction weights for the example gene
twas_weight = readRDS('output/finemapping/susie_twas/twas_weights/protocol_example_susie_twas.chr22_ENSG00000283047.univariate_twas_weights.rds')
str(twas_weight, max.level = 2)

List of 1
 $ ENSG00000283047:List of 1
  ..$ bulk_rnaseq_ENSG00000283047:List of 9


In [17]:
# R kernel
# Extract the fitted weights by method for the gene/context
twas_weight_by_method = twas_weight$ENSG00000283047$bulk_rnaseq_ENSG00000130538$twas_weights
str(twas_weight_by_method)

 NULL


### **Step 2.** Run TWAS and Mendelian Randomization


In [ ]:
sos run pipeline/twas_ctwas.ipynb twas \
    --cwd output/twas \
    --name protocol_example \
    --gwas_meta_data input/twas/protocol_example.twas.gwas_meta.tsv \
    --ld_meta_data input/LD_sketch/meta.tsv \
    --ld_reference_sample_size 60 \
    --regions input/twas/protocol_example.twas.LD_blocks.chr22.bed \
    --xqtl_meta_data input/twas/protocol_example.twas.xqtl_meta.tsv \
    --xqtl_type_table input/twas/protocol_example.twas.data_type_table.txt \
    --rsq_pval_cutoff 1 \
    --rsq_cutoff 0.01

### Inspect the TWAS and MR results


In [ ]:
# Inspect the TWAS association results
zcat output/twas/twas/protocol_example.chr22_10000000_19000000.twas.tsv.gz | head

In [ ]:
# Inspect the Mendelian Randomization (MR) results
zcat output/twas/twas/protocol_example.chr22_10000000_19000000.mr_result.tsv.gz | head

### Output Files

| File | Contents |
|------|----------|
| `output/twas/twas/protocol_example.chr22_*.twas.tsv.gz` | Per-gene TWAS Z-scores and p-values |
| `output/twas/twas/protocol_example.chr22_*.mr_result.tsv.gz` | Mendelian Randomization results for candidate genes |

## 2. xQTL–GWAS enrichment

This command executes the **`xqtl_gwas_enrichment`** workflow, which analyzes enrichment relationships between xQTL and GWAS data

#### **Purpose**

The purpose of this step is to calculate enrichment parameters (a0, a1) between xQTL and GWAS data and generate prior probabilities (p1, p2, p12) for subsequent colocalization analysi

#### **Method**

The workflow uses the **`xqtl_enrichment_wrapper`** function to perform enrichment analysis . The method:

1. Identifies GWAS blocks with top loci using single variant regression methods
2. Maps analysis regions to overlapping gene regions with corresponding QTL files containing top loci tables
3. Finds contexts within xQTL metadata that include top loci results for each gene
4. Executes enrichment analysis to generate parameters SuSiE_enloc.ipynb:102-106

#### **Input Data**

**Metadata files:**

- **`-gwas-meta-data`**: Metadata file for GWAS fine-mapping results
- **`-xqtl-meta-data`**: Metadata file for xQTL fine-mapping results
- **`-context_meta`**: Meta file showing analysis names and contained contexts

**Data paths:**

- **`-qtl-path`** and **`-gwas_path`**: Directory paths for original fine-mapping results

#### **Parameter Interpretation**

**Object access parameters:**

- **`-xqtl_finemapping_obj`**: Table name in xQTL RDS files to get fine-mapping results
- **`-xqtl_varname_obj`**: Table name to get variable names
- **`-gwas_finemapping_obj`** and **`-gwas_varname_obj`**: Corresponding parameters for GWAS data
- **`-xqtl_region_obj`**: Table name to get region information

#### **Output**

Generates enrichment analysis result files: **`*.enrichment.rds`**, containing global enrichment estimates that combine all input data for each context. The output file path format is: **`{cwd}/{name}.{context}.enrichment.rds`** 

#### **Relationship to Downstream Analysis**

This enrichment analysis is a prerequisite step for colocalization analysis. The generated enrichment parameters serve as prior probability inputs for the **`susie_coloc`** workflow. In colocalization analysis, the system automatically reads enrichment result files to set p1, p2, and p12 prior probabilities.

#### **Notes**

This workflow is the first stage of the SuSiE-enloc framework, specifically handling enrichment analysis between fine-mapping results from different genomic regions. The workflow automatically handles region overlap identification and context matching, providing statistically sound prior probabilities for subsequent colocalization analysis.

### Input Files

| File | Produced by | Used as |
|------|-------------|---------|
| `input/susie_enloc_data/protocol_example.enloc.gwas_meta.tsv` | SuSiE-enloc demo | GWAS fine-mapping metadata |
| `input/susie_enloc_data/protocol_example.enloc.xqtl_meta.tsv` | SuSiE-enloc demo | xQTL fine-mapping metadata |
| `input/susie_enloc_data/protocol_example.enloc.context_meta.tsv` | SuSiE-enloc demo | context / analysis-name map |
| `input/susie_enloc_data/*` | SuSiE-enloc demo | fine-mapping result objects |


### **Step 1.** Run xQTL–GWAS enrichment analysis


In [ ]:
sos run pipeline/SuSiE_enloc.ipynb xqtl_gwas_enrichment \
    --gwas-meta-data input/susie_enloc_data/protocol_example.enloc.gwas_meta.tsv \
    --xqtl-meta-data input/susie_enloc_data/protocol_example.enloc.xqtl_meta.tsv \
    --xqtl_finemapping_obj preset_variants_result susie_result_trimmed \
    --xqtl_varname_obj preset_variants_result variant_names \
    --gwas_finemapping_obj AD_Bellenguez_2022 RSS_QC_RAISS_imputed susie_result_trimmed \
    --gwas_varname_obj AD_Bellenguez_2022 RSS_QC_RAISS_imputed variant_names \
    --xqtl_region_obj region_info grange \
    --qtl-path input/susie_enloc_data \
    --gwas-path input/susie_enloc_data \
    --context-meta input/susie_enloc_data/protocol_example.enloc.context_meta.tsv \
    --cwd output/xqtl_gwas_enrichment


### Inspect the enrichment parameters


In [ ]:
# Inspect the enrichment parameters between xQTL and GWAS
# R kernel
coloc_enrichment = readRDS('output/xqtl_gwas_enrichment/protocol_example.enloc.Knight_eQTL_brain.enrichment.rds')
str(coloc_enrichment)

### Output Files

| File | Contents |
|------|----------|
| `output/xqtl_gwas_enrichment/*.enrichment.rds` | Enrichment parameters (a0, a1) and derived colocalization priors (p1, p2, p12) |

## 3. Pairwise colocalization (SuSiE-coloc)

This command executes the **`susie_coloc`** workflow, which performs colocalization analysis between xQTL and GWAS data to identify shared causal variants.

#### **Purpose**

The purpose of this step is to perform pairwise colocalization analysis between xQTL and GWAS fine-mapping results to determine whether observed associations share causal variants in overlapping genomic regions. This analysis identifies contexts with top loci results for each gene and applies **`susie_coloc`** to analyze each gene under each identified condition.

#### **Method**

The workflow uses the **`coloc_wrapper`** function to perform colocalization analysis. The method:

1. **Region Overlap Detection**: Identifies GWAS blocks that contain overlapping top loci variants for each gene using genomic coordinate matching
2. **Prior Probability Setting**: Either loads enrichment-derived priors from previous analysis or uses default values when **`-skip-enrich`** is specified
3. **Colocalization Analysis**: Applies **`coloc_wrapper`** for each xQTL-GWAS file pair with appropriate prior probabilities

#### **Input Data**

**Metadata files:**

- **`-gwas-meta-data`**: GWAS fine-mapping results metadata
- **`-xqtl-meta-data`**: xQTL fine-mapping results metadata with overlap information
- **`-context_meta`**: Analysis names and context mappings

**Data paths:**

- **`-qtl-path`** and **`-gwas_path`**: Directory paths for original fine-mapping results

#### **Parameter Interpretation**

**Object access parameters:**

- **`-xqtl_finemapping_obj`**: Table name in xQTL RDS files for fine-mapping results
- **`-xqtl_varname_obj`**: Table name for variable names
- **`-gwas_finemapping_obj`** and **`-gwas_varname_obj`**: Corresponding GWAS parameters
- **`-xqtl_region_obj`**: Table name for region information

**Analysis control parameters:**

- **`-skip-enrich`**: Skips enrichment analysis and uses default prior probabilities (p1=1e-4, p2=1e-4, p12=5e-6)
- **`-ld_meta_file_path`**: LD reference metadata for post-processing credible sets

#### **Output**

Generates colocalization result files: **`*.coloc.rds`** containing colocalization statistics and posterior probabilities for each gene-context pair. When LD metadata is provided, also outputs credible set files (**`*.coloc_res`**) with variant-level results SuSiE_enloc.ipynb:851 .

#### **Notes**

This workflow represents the second stage of the SuSiE-enloc framework, performing gene-by-gene colocalization analysis on regions identified to have overlapping variants. The **`--skip-enrich`** parameter allows bypassing enrichment analysis when using default priors, making the analysis faster but potentially less statistically informed. The inclusion of LD metadata enables variant-level credible set reporting for significant colocalization results.

### Input Files

| File | Produced by | Used as |
|------|-------------|---------|
| `input/susie_enloc_data/protocol_example.enloc.gwas_meta.tsv` | SuSiE-enloc demo | GWAS fine-mapping metadata |
| `input/susie_enloc_data/protocol_example.enloc.xqtl_meta.tsv` | SuSiE-enloc demo | xQTL fine-mapping metadata (with overlap info) |
| `input/susie_enloc_data/protocol_example.enloc.context_meta.tsv` | SuSiE-enloc demo | context / analysis-name map |
| `input/ld_reference/protocol_example.ld_meta_file.tsv` | toy data prep | LD reference metadata for credible sets |
| `input/susie_enloc_data/*` | SuSiE-enloc demo | fine-mapping result objects |


### **Step 1.** Run pairwise colocalization (SuSiE-coloc)


In [ ]:
sos run pipeline/SuSiE_enloc.ipynb susie_coloc \
    --gwas-meta-data input/susie_enloc_data/protocol_example.enloc.gwas_meta.tsv \
    --xqtl-meta-data input/susie_enloc_data/protocol_example.enloc.xqtl_meta.tsv \
    --xqtl_finemapping_obj preset_variants_result susie_result_trimmed \
    --xqtl_varname_obj preset_variants_result variant_names \
    --gwas_finemapping_obj AD_Bellenguez_2022 RSS_QC_RAISS_imputed susie_result_trimmed \
    --gwas_varname_obj AD_Bellenguez_2022 RSS_QC_RAISS_imputed variant_names \
    --xqtl_region_obj region_info grange \
    --qtl-path input/susie_enloc_data \
    --skip-enrich \
    --gwas-path input/susie_enloc_data \
    --context-meta input/susie_enloc_data/protocol_example.enloc.context_meta.tsv \
    --ld_meta_file_path input/ld_reference/protocol_example.ld_meta_file.tsv \
    --cwd output/susie_coloc


### Inspect the colocalization results


In [ ]:
# Inspect the colocalization results for one gene
# R kernel
coloc_result = readRDS('output/susie_coloc/susie_coloc/protocol_example.enloc.Knight_eQTL_brain@ENSG00000142798.coloc.rds')
str(coloc_result)

### Output Files

| File | Contents |
|------|----------|
| `output/susie_coloc/susie_coloc/*@<gene>.coloc.rds` | Per gene-context colocalization stats and posterior probabilities |
| `output/susie_coloc/*.coloc_res` | Variant-level credible-set results (when LD metadata is supplied) |

Each `.coloc.rds` is a nested R list with `summary` (posterior probabilities for the five coloc hypotheses), `results` (SNP-level PP.H4), `priors`, and `analysis_region`. **PP.H4** is the probability that both traits share a single causal variant — higher values mean stronger colocalization evidence.

Each `.coloc.rds` reports posterior probabilities for five mutually exclusive colocalization hypotheses for a given gene-trait pair: **PP.H0** (neither trait has an association in the region), **PP.H1** (only the xQTL is associated), **PP.H2** (only the GWAS trait is associated), **PP.H3** (both are associated but driven by *different* causal variants), and **PP.H4** (both are associated and share a *single* causal variant). A high **PP.H4** is the evidence for colocalization — it indicates the molecular and complex traits are likely driven by the same underlying variant.

#### Interpretation of `coloc` Analysis Results (Gene: `ENSG00000142798`)

##### 1). Overall Structure

The provided object is a nested list in R, structured specifically for interpreting coloc analysis results. It contains five main components:

- `summary`: Posterior probabilities for different coloc hypotheses.
- `results`: SNP-level posterior probabilities (PP.H4).
- `priors`: Priors used in Bayesian coloc analysis.
- `analysis_region`: Genomic region analyzed.
- `sets`: Credible sets; here `NULL`, indicating no credible set was formed.

---

##### 2). `summary` Table

This table contains posterior probabilities for five coloc hypotheses across SNP pairs:

- `nsnps`: Number of SNPs analyzed (8826).
- `hit1`: Representative SNP for signal 1.
- `hit2`: Representative SNP for signal 2.

Five coloc hypotheses probabilities:

- `PP.H0.abf`: Neither trait has an association (null hypothesis).
- `PP.H1.abf`: Only trait 1 is associated.
- `PP.H2.abf`: Only trait 2 is associated.
- `PP.H3.abf`: Both traits are associated, but at different SNPs.
- `PP.H4.abf`: Both traits are associated at the same SNP (colocalization).

Higher `PP.H4.abf` indicates stronger evidence for colocalization.

---

##### 3). `results` Table

Contains posterior probabilities for each of the 8826 SNPs across ten coloc scenarios (`rows`):

- `snp`: SNP identifier (e.g., `"1:20520243:G:A"`).
- `SNP.PP.H4.row1` to `SNP.PP.H4.row10`: SNP-specific posterior probabilities for colocalization scenarios corresponding to each row in `summary`.

High SNP-level posterior probabilities indicate likely colocalization positions.

---

##### 4). `priors`

Prior probabilities used in Bayesian analysis:

- `p1`: Prior probability that trait 1 is associated.
- `p2`: Prior probability that trait 2 is associated.
- `p12`: Prior probability of colocalization between trait 1 and trait 2.

Typically set low, reflecting conservative assumptions.

---

##### 5). `analysis_region`

Genomic region analyzed (e.g., `"chr1:20520000-24080000"`).

---

##### 6). `sets`

Contains credible sets (`cs`):

- Here `cs` is `NULL`, indicating no credible set formed, possibly due to weak evidence.

---

##### 7). Guidelines for Interpretation

Interpretation of `PP.H4.abf`:

- `PP.H4.abf > 0.75`: Strong evidence of colocalization.
- `0.25 ≤ PP.H4.abf ≤ 0.75`: Moderate evidence of colocalization.
- `PP.H4.abf < 0.25`: Weak evidence of colocalization.

When `PP.H4.abf` is high, further examine individual SNP-level probabilities (`results`) to pinpoint the SNP responsible for the shared signal.

---

##### 8). Practical Example

For row 2 of the `summary`:

- `hit1`: `1:21912282:C:G`
- `hit2`: `1:22187644:C:A`
- `PP.H4.abf`: 0.02221 (~2.2%)

The low `PP.H4.abf` indicates weak colocalization evidence. The higher `PP.H3.abf` (~40%) implies distinct SNP associations.

Inspect `results` for high individual SNP posterior probabilities (e.g., > 0.9) to identify candidate colocalized SNPs.

---

##### 9). Summary of Key Points

- Inspect `summary` for overall colocalization evidence (`PP.H4.abf`).
- Check `results` for SNP-level posterior probabilities to identify potential colocalized SNPs.
- A `NULL` credible set indicates insufficient evidence to confidently define a credible SNP set.

## 4. ColocBoost

**ColocBoost** is a multi-trait colocalization analysis tool designed to identify shared genetic variants influencing multiple molecular traits.


### Core Features

- Performs colocalization analysis within genomic regions accounting for **multiple causal variants**
- Scales to the analysis of **hundreds of traits**
- Supports inclusion or exclusion of **GWAS summary statistics**
- Requires **individual-level xQTL data from the same cohort**

### Analysis Modes

ColocBoost supports three major analysis modes:

- xQTL-only Analysis (`--xqtl-coloc`):  
  Performs colocalization only among molecular traits.

- Joint GWAS Analysis (`--joint-gwas`):  
  Integrates all GWAS traits in a combined colocalization analysis.

- Separate GWAS Analysis (`--separate-gwas`):  
  Performs colocalization analysis separately for each GWAS trait.

### `--separate-gwas` vs `--no-separate-gwas`

#### `--separate-gwas` (Enabled by default)

When this flag is active, ColocBoost performs **individual colocalization analyses** for each GWAS trait.

- Generates **one output file per GWAS study**
- Output format: `{base_filename}.cb_xqtl_{study_name}.rds`
- Supports **trait-specific** analysis and result inspection

#### `--no-separate-gwas`

When this flag is used, separate GWAS analyses are **not performed**.

- Skips individual trait analysis
- Reduces computational burden and the number of output files
- Typically used **in combination with** `--joint-gwas`


### Practical Examples

#### xQTL-only Analysis

- Use `--no-separate-gwas` to focus solely on **colocalization between molecular traits**

#### GWAS-xQTL Joint Analysis

- Use `--separate-gwas` to perform **trait-specific** colocalization for each GWAS dataset

### Input Files

| File | Produced by | Used as |
|------|-------------|---------|
| `output/plink/protocol_example.genotype.merged.plink_qc.bed` | genotype preprocessing | merged genotypes |
| `output/phenotype/.../bulk_rnaseq.phenotype_by_chrom_files.region_list.txt` | phenotype preprocessing | phenotype region list |
| `output/covariate/protocol_example...Marchenko_PC.gz` | covariate preprocessing | covariates |
| `input/reference_data/TAD/TADB_enhanced_cis.bed` | reference data | cis association windows |
| `input/LD_sketch/meta.tsv` | RSS LD sketch step | LD reference metadata (joint mode) |
| `input/colocboost/gwas_meta.txt` | toy data prep | GWAS metadata, tab-separated (joint mode) |


### **Step 1.** xQTL-only ColocBoost (`--no-separate-gwas`)


In [ ]:
sos run pipeline/colocboost.ipynb colocboost \
    --name test_coloc_boost_xqtl_only \
    --cwd output/colocboost \
    --genoFile output/plink/protocol_example.genotype.merged.plink_qc.bed \
    --phenoFile output/phenotype/phenotype_by_chrom_for_cis/bulk_rnaseq.phenotype_by_chrom_files.region_list.txt \
    --covFile output/covariate/protocol_example.rnaseq.bed.protocol_example.covariates.protocol_example.genotype.merged.plink_qc.plink_qc.prune.pca.Marchenko_PC.gz \
    --customized-association-windows input/reference_data/TAD/TADB_enhanced_cis.bed \
    --no-separate-gwas --xqtl-coloc \
    --region-name ENSG00000130538 \
    --phenotype-names trait_A


### **Step 2.** Joint xQTL–GWAS ColocBoost (`--separate-gwas`)


In [ ]:
sos run pipeline/colocboost.ipynb colocboost \
    --name colocboost_gwas \
    --cwd output/colocboost_gwas \
    --genoFile output/plink/protocol_example.genotype.merged.plink_qc.bed \
    --phenoFile output/phenotype/phenotype_by_chrom_for_cis/bulk_rnaseq.phenotype_by_chrom_files.region_list.txt \
    --covFile output/covariate/protocol_example.rnaseq.bed.protocol_example.covariates.protocol_example.genotype.merged.plink_qc.plink_qc.prune.pca.Marchenko_PC.gz \
    --customized-association-windows input/reference_data/TAD/TADB_enhanced_cis.bed \
    --ld-meta-data input/LD_sketch/meta.tsv \
    --gwas-meta-data input/colocboost/gwas_meta.txt \
    --separate-gwas --xqtl-coloc \
    --region-name ENSG00000130538 \
    --phenotype-names trait_A


### Inspect the ColocBoost output


In [ ]:
# Inspect the ColocBoost output
# R kernel
colocboost = readRDS('output/colocboost_gwas/colocboost/colocboost_gwas.chr22_ENSG00000130538.cb_xqtl_synthetic_gwas_chr22.rds')
str(colocboost)

### Output Files

| File | Contents |
|------|----------|
| `output/colocboost/.../cb_xqtl_*.rds` | xQTL-only ColocBoost result per gene |
| `output/colocboost_gwas/.../cb_xqtl_synthetic_gwas_*.rds` | Joint xQTL–GWAS ColocBoost result per gene |

Each `.rds` is a named list (class `colocboost`) keyed by gene. The most useful elements are `cos_summary` (colocalized signals, empty if none), `cos_details` (details per colocalized signal), `ucos_details` (uncolocalized signals — many of these suggest GWAS signals independent of molecular QTLs), and `data_info` / `model_info` (traits analyzed and model convergence).

### Colocboost Output File Structure

Each `.rds` holds the `colocboost` result for one gene: a named list of class `"colocboost"` whose top-level element corresponds to that gene (e.g. `ENSG00000049246`). In joint xQTL–GWAS mode (`--separate-gwas`) one such file is written per GWAS trait, named `{base_filename}.cb_xqtl_{study_name}.rds`. The internal structure is organized into the components below.

#### Top-Level Components

| Field | Type | Description |
| --- | --- | --- |
| `cos_summary` | NULL or list | Summary of colocalized signals (empty if none) |
| `vcp` | NULL or list | Variant colocalization probabilities |
| `cos_details` | NULL or list | Details for colocalized signals |
| `data_info` | list | Information on traits, SNPs, z-scores, and coefficients |
| `model_info` | list | Model convergence and log-likelihood tracking |
| `ucos_details` | list | Results for **uncolocalized signals** (unique to colocboost) |
| `region_info` | list | Genomic coordinates and extended window used in analysis |
| `computing_time` | list | Timing for loading, QC, and model fitting steps |

---

#### data_info

| Field | Description |
| --- | --- |
| `n_outcomes` | Number of traits analyzed |
| `n_variables` | Number of SNPs analyzed |
| `outcome_info` | Data frame of each trait’s name, sample size, is_sumstats, and is_focal |
| `variables` | Character vector of SNP IDs in `chr:pos:ref:alt` format |
| `coef`, `z` | Trait-wise regression coefficients and z-scores (lists with length = number of traits) |

---

#### model_info

Tracks model convergence and optimization metrics for each trait:

| Field | Description |
| --- | --- |
| `model_coveraged` | Whether the joint model converged |
| `outcome_model_coveraged` | Per-trait convergence status |
| `n_updates` | Total number of updates |
| `outcome_n_updates` | Number of updates per trait |
| `profile_loglik` | Log-likelihood per iteration (joint) |
| `outcome_profile_loglik` | Log-likelihood per trait |
| `outcome_proximity_obj` | Proximity statistics per trait per iteration |
| `outcome_coupled_best_update_obj` | Coupled update quality per trait |
| `jk_star` | Jackknife matrix (optional) |

---

#### ucos_details

Describes **uncolocalized effect sets** when no coloc is detected:

| Field | Description |
| --- | --- |
| `ucos_index` | Indices of SNPs in each uncolocalized set |
| `ucos_variables` | SNP IDs in each set |
| `ucos_outcomes` | Trait name associated with each set (e.g., `"Wightman"`) |
| `ucos_weight` | Per-SNP weights in each set |
| `ucos_top_variables` | Top SNP per set |
| `ucos_purity` | Pairwise correlation (min, max, median) across sets |
| `ucos_outcomes_delta` | Delta value (importance) per set |

> Interpretation:
> 
> - Large number of `ucos` suggests GWAS has signals independent of molecular QTLs.
> - `ucos_delta` indicates signal strength; higher = more trait variance explained.

---

#### region_info

| Field | Description |
| --- | --- |
| `region_coord` | Original gene start/end coordinates |
| `grange` | Extended region used for coloc (typically ±1.5Mb) |
| `region_name` | Gene or region ID (can repeat if multiple traits mapped) |

#### computing_time
---


#### Key Results Interpretation in this MWE

#### Analysis Overview

- **Gene**: ENSG00000049246 (chromosome 1)
- **Traits analyzed**:  
  - `trait_A_ENSG00000049246` (xQTL, n = 150)  
  - `Wightman` (GWAS, n = 363,646)
- **Variants**: 6,872 variants analyzed within the region

#### Missing Core Results

- `cos_summary`: `NULL` – No colocalized effects detected  
- `vcp`: `NULL` – No variant colocalization probabilities computed  
- `cos_details`: `NULL` – No detailed colocalization results available

This indicates no significant colocalization between xQTL and GWAS signals for the target gene.

These results suggest that while the Wightman GWAS shows multiple independent association signals in the region, these signals do not colocalize with the xQTL effects. 

### Uncolocalized Effects (`ucos_details`)

The analysis detected 11 uncolocalized effects (`ucos1:y2` to `ucos11:y2`), all associated with the GWAS trait `Wightman`.  

- **Top variants**:  
  - `chr1:9797684:A:C`  
  - `chr1:9809863:C:T`  
  - `chr1:9830837:C:T`  
- **Effect strengths**:  
  - Ranged from extremely weak (`ucos9:y2` with weights ≈ 9e-13)  
  - To moderate (`ucos5:y2` with weights ≈ 3e-06)  
- **Delta values**:  
  - Effect size contributions ranged from 0.0088 to 0.0743

### Model Convergence

- `trait_A_ENSG00000049246`: Converged after 20 updates (`model_coveraged = TRUE`)
- `Wightman`: Did not converge after 501 updates (`model_coveraged = FALSE`)


### Technical Details

- **Analysis region**: chr1:6,120,000–9,960,000  
- **Gene coordinates**: chr1:7,784,319–7,845,176  

**Computing time**:  
- Data loading: ~3 minutes  
- Quality control: ~17 seconds  
- Model fitting:  
  - xQTL: ~15 seconds  
  - GWAS integration: ~4.7 minutes